In [ ]:
import requests, json

# vLLM health
print("vLLM health:", requests.get("http://localhost:8000/health", timeout=5).text)

# vLLM models
print("vLLM models:", requests.get("http://localhost:8000/v1/models", timeout=10).json())


In [ ]:
import requests
r = requests.get("http://localhost:55678", timeout=5)
print("Neo4j HTTP status:", r.status_code)


In [ ]:
from neo4j import GraphDatabase

URI = "bolt://localhost:55677"

# If auth disabled:
AUTH = None

# If auth enabled, use:
# AUTH = ("neo4j", "YOUR_PASSWORD")

driver = GraphDatabase.driver(URI, auth=AUTH) if AUTH else GraphDatabase.driver(URI)

cypher = """
MERGE (p:Person {name:$name})
MERGE (t:Topic {name:$topic})
MERGE (p)-[:INTERESTED_IN]->(t)
RETURN p,t
"""
with driver.session() as s:
    rec = s.run(cypher, name="You", topic="Knowledge Graphs").single()
    print(rec["p"], rec["t"])


In [ ]:
from openai import OpenAI

client = OpenAI(
    base_url="http://localhost:8000/v1",
    api_key=""  # vLLM ignores it
)

resp = client.chat.completions.create(
    model="Qwen/Qwen2.5-3B-Instruct",
    messages=[
        {"role": "system", "content": "You are a helpful assistant."},
        {"role": "user", "content": "Write 5 Cypher queries to explore a graph of papers and authors."}
    ],
    temperature=0.2,
)
print(resp.choices[0].message.content)


In [ ]:
from openai import OpenAI
from neo4j import GraphDatabase

client = OpenAI(base_url="http://localhost:8000/v1", api_key="")
driver = GraphDatabase.driver("bolt://localhost:55677")  # add auth=... if needed

schema_hint = """
Graph schema:
(:Person {name})
(:Topic {name})
(:Person)-[:INTERESTED_IN]->(:Topic)
"""

question = "Show all people and the topics they are interested in."

prompt = f"""{schema_hint}
Write ONE Cypher query that answers: {question}
Return columns: person, topic
Only output the Cypher query, nothing else.
"""

cypher = client.chat.completions.create(
    model="Qwen/Qwen2.5-3B-Instruct",
    messages=[{"role":"user","content":prompt}],
    temperature=0.0,
).choices[0].message.content.strip()

print("Generated Cypher:\n", cypher)

with driver.session() as s:
    rows = s.run(cypher).data()
rows


In [ ]:
# Step 1 – design a schema
!python  jupyter/graphrag/schema_design.py --text "We propose GraphFormer... evaluate on SciERC..." --out schema.json



In [ ]:
# Step 2 – ingest documents
!python jupyter/graphrag/ingest.py --schema schema.json --docs  "We propose GraphFormer... evaluate on SciERC..."


In [ ]:
!python jupyter/graphrag/query.py --schema schema.json --question "Which methods are most commonly used, and on which datasets are they evaluated?"
